In [1]:
import numpy as np
import pandas as pd
import torch
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
import pymorphy3
import re
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from rank_bm25 import BM25Okapi
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "sergeyzh/rubert-mini-retriever" #"deepvk/USER-bge-m3"
CROSS_ENCODER_NAME = "DiTy/cross-encoder-russian-msmarco"
TOP_K = 10
TOP_N = 20
THRESHOLD_MAP = 0.1
THRESHOLD_DF = 0.8
TITLE_WEIGHT = 2
W_EMB = 0.55
W_BM25 = 0.45
W_CALIB = 1

print(DEVICE)

cuda


In [2]:
model = SentenceTransformer(MODEL_NAME, device = DEVICE)
cross_encoder = None
#cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device = DEVICE)

In [3]:
articles = pd.read_feather("candidate_public/candidate_data/articles.f")
calibration = pd.read_feather("candidate_public/candidate_data/calibration.f")
test = pd.read_feather("candidate_public/candidate_data/test.f")

1) Инициализировать БД для статей нецелесообразно, не выигрываем по вычислениям на 793 статьи; работаем со списком обычным, но оберну его в объект самописный, чтобы модель скачивалась и подгружалась только при инициализации, а не при каждом вызове функции. Ну и гибкости будто добавит.
2) Обработка обязательна, разметка будет влиять на embedding. Применять буду на всей базе, добавляя новый столбец к данным. Baseline-cвязка BeautifulSoup+NLTK.
3) В голове идея попробовать сначала базовый пайплайн с простым эмбеддером, потом поменять на побольше, потом попробовать топ-50 и rerank с каким-то cross-encoder
4) Разбиваю пайплайн на функциия нарочно, буду менять составляющие по ходу экспериментов.

In [4]:
morph = pymorphy3.MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))
russian_stopwords.update(['это','так','все','или','и',"в","на","с","по","не","что","как","это","для","от","из",
                          "за","к","о","а","но","бы","же","ли","ни","до","при","во","со","без","под","над","об",
                          "его","её","их","мы","вы","он","она","они","я","ты","то","всё","уже","ещё","нет","да",
                          "если","или","когда","где","кто","чем","тем","этом","этой","этого","того","здравствуйте",
                          "добрый","день","вечер"
                         ])
def html_cleaner(s:str) -> str:
    soup = BeautifulSoup(s, 'html.parser')
    text = soup.get_text(separator=' ')
    text = re.sub(r'[^a-zа-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).lower().strip()
    return text

def body_cleaner(s: str, extra_stopwords=None) -> str:
    """
    Cleans the body string (deleted html-tags, stopwords; lowered, lemmatized).
    
    Args:
        s (string): The string to clean

    Returns:
        string: Clear string.
    """
    text = html_cleaner(s)
    tokens = text.split()
    lemmatized = []
    for token in tokens:
        if token not in russian_stopwords and len(token) > 1:
            normal = morph.parse(token)[0].normal_form
            if extra_stopwords and normal in extra_stopwords:
                continue
            lemmatized.append(normal)
    return ' '.join(lemmatized)
    
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full DataFrame preprocess. Initializes new column - 'processed' (string), ready to encode.

    Args:
        df (pd.DataFrame): The source dataset. No actions are made.

    Returns:
        pd.DataFrame: The same dataset with stacked clear column 'processed'.
    """
    df['processed'] = df['body']
    df['processed'] = df['processed'].apply(body_cleaner)
    df['html_processed'] = articles['title'] + ' ' + df['body']
    df['html_processed'] = df['html_processed'].apply(html_cleaner)
    all_tokens = []
    for text in df['processed']:
        all_tokens.extend(text.split())
    doc_freq = Counter()
    for text in df['processed']:
        unique_tokens = set(text.split())
        doc_freq.update(unique_tokens)
    N_docs = len(df)
    high_freq_words = {word for word, df in doc_freq.items() if df / N_docs > THRESHOLD_DF}
    df['processed'] = df['processed'].apply(lambda x: body_cleaner(x, extra_stopwords=high_freq_words))
    
    return df

In [5]:
class Database:
    def __init__(self, df: pd.DataFrame, document_preprocess, query_preprocess, model, cross_encoder):
        """
        Database - the orchestrator of the full pipeline
        Args:
            df (pd.DataFrame): The source dataset. No actions are made.
        """
        self.model = model
        self.cross_encoder = cross_encoder
        self.processed_articles = document_preprocess(df)
        self.article_embeddings = model.encode(
            self.processed_articles['processed'].tolist(),
            show_progress_bar=True,
            convert_to_numpy=True
        ) + TITLE_WEIGHT * model.encode(
            self.processed_articles['title'].tolist(),
            show_progress_bar=True,
            convert_to_numpy=True
        )
        tokenized_docs = [doc.split() for doc in self.processed_articles['html_processed']]
        self.bm25 = BM25Okapi(tokenized_docs)
        self.query_preprocess = query_preprocess
    def retrieve(self, query: str, top_k: int = TOP_K) -> list:
        """
        Returns the list of relevant articles` indicies (top-k, descending relevance sorted).

        Args:
            query (string): The user`s request to the system.
            top_k (int): The number of the most relevent values to return.

        Returns: 
            list: sorted article_ids of the relevant articles
        """
        query_processed = self.query_preprocess(query)
        query_emb = model.encode([query_processed], convert_to_numpy = True)
        similarities = cosine_similarity(query_emb, self.article_embeddings)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return self.processed_articles.iloc[top_indices]['article_id'].astype(str).tolist()

    def retrieve_hybrid(self, query: str, top_k: int = TOP_K, top_n:int = TOP_N) -> list:
        query_processed = self.query_preprocess(query)
        emb_scores = None
        if query_processed.strip():
            query_emb = self.model.encode([query_processed], convert_to_numpy=True)
            emb_scores = cosine_similarity(query_emb, self.article_embeddings)[0]
        else:
            emb_scores = np.zeros(len(self.processed_articles))
    
        bm25_query = html_cleaner(query)
        tokenized_query = bm25_query.split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
    
        # нормализую оба набора скоров (min‑max)
        def normalize(scores):
            min_s, max_s = scores.min(), scores.max()
            if max_s == min_s:
                return np.ones_like(scores)
            return (scores - min_s) / (max_s - min_s)
    
        emb_norm = normalize(emb_scores)
        bm25_norm = normalize(bm25_scores)
    
        # Комбинированный скор
        weight_emb = W_EMB
        weight_bm25 = W_BM25
        combined = weight_emb * emb_norm + weight_bm25 * bm25_norm
    
        top_indices = np.argsort(combined)[::-1][:top_k]
        return self.processed_articles.iloc[top_indices]['article_id'].astype(str).tolist()
    
    def retrieve_reranked_with_bm25(self, query: str, top_k: int = TOP_K, top_n:int = TOP_N) -> list:
        query_processed = self.query_preprocess(query)
        emb_indices = []
        if query_processed.strip():
            query_emb = self.model.encode([query_processed], convert_to_numpy=True)
            similarities = cosine_similarity(query_emb, self.article_embeddings)[0]
            emb_indices = np.argsort(similarities)[::-1][:top_n]
    
        bm25_query = html_cleaner(query)
        tokenized_query = bm25_query.split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_indices = np.argsort(bm25_scores)[::-1][:top_n]
    
        combined_indices = list(set(list(emb_indices) + list(bm25_indices)))
        candidate_df = self.processed_articles.iloc[combined_indices]
    
        pairs = [(query, row['html_processed']) for _, row in candidate_df.iterrows()]
        scores = self.cross_encoder.predict(pairs)
        sorted_idx = np.argsort(scores)[::-1][:top_k]
        result_ids = candidate_df.iloc[sorted_idx]['article_id'].astype(str).tolist()
        return result_ids
    
    def _ap_at_10(self, actual, predicted, k=TOP_K):
        """
        Metric AP@k

        Args:
            ground_true (list): true labels 
            predicted (list): predicted labels
        Returns:
            float: MetricValue
        """
        assert isinstance(actual, list)
        
        assert (isinstance(actual[0], str) == isinstance(predicted[0], str))
        if not actual:
            return 0.0
    
        if len(predicted) > k:
            predicted = predicted[:k]
    
        score = 0.0
        num_hits = 0.0
    
        for i, p in enumerate(predicted):
            if p in actual and p not in predicted[:i]:
                num_hits += 1.0
                score += num_hits / (i + 1.0)
    
        return score / min(len(actual), k)
        
    def calibrate(self, calibration: pd.DataFrame) -> float:
        """
        Returns MAP@10 for calibration dataset.

        Args:
            calibration (pd.DataFrame): The dataset to calibrate on (labels are guaranteed).

        Returns:
            float: The metric value.
        """
        calibration['predicted'] = calibration['query_text'].apply(self.retrieve_hybrid)
        calibration['ap'] = calibration.apply(
            lambda row: self._ap_at_10(row['ground_truth'].split(), row['predicted'], TOP_K),
            axis=1
        )
        map_score = calibration['ap'].mean()

        bad_cases = calibration[calibration['ap'] < THRESHOLD_MAP].copy()
        bad_cases = bad_cases[['query_id', 'query_text', 'ground_truth', 'predicted', 'ap']]
        bad_cases.to_csv("bad_cases.csv", index=False)
        print(f"Найдено {len(bad_cases)} запросов с AP < {THRESHOLD_MAP}. Данные сохранены в bad_cases.csv")
        return map_score, bad_cases

    def answer(self, test: pd.DataFrame) -> pd.DataFrame:
        test['answer'] = test['query_text'].apply(lambda q: ' '.join(self.retrieve_hybrid(q)))
        answer = test[['query_id', 'answer']]
        answer.to_csv("answer.csv", index=False)

In [6]:
db = Database(
    articles,
    preprocess,
    body_cleaner,
    model,
    cross_encoder
)
MAP, bads = db.calibrate(calibration)
print(MAP)
db.answer(test)

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

/tmp/ipykernel_2352/2140867575.py:10: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(s, 'html.parser')
/tmp/ipykernel_2352/2140867575.py:10: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(s, 'html.parser')


Найдено 135 запросов с AP < 0.1. Данные сохранены в bad_cases.csv
0.38092261904761904
